# Practical 1 — Tokenizers, embeddings et attention multi-tête (correction commentée)

Ce TP construit, étape par étape, **l'entrée d'un modèle GPT-2** et son cœur : **l'attention**.

Le chemin complet d'un texte dans GPT ressemble à ceci :

```
texte  ──tokenizer──▶  ids (nombres)  ──embedding──▶  vecteurs  ──+ position──▶  attention  ──▶ ...
"Hello"               [15496]                         [0.3, -1.2, ...]
```

Plan du notebook :
1. **Tokenizer** : découper le texte en morceaux (tokens) et les transformer en nombres.
2. **Dataset / DataLoader** : préparer des paires (entrée, cible) pour apprendre à prédire le mot suivant.
3. **Embeddings de mots** : transformer chaque nombre en vecteur.
4. **Embeddings de position** : indiquer au modèle *où* est chaque mot.
5. **Attention** à la main, puis en classe `SelfAttention`, `CausalAttention`, `MultiHeadAttention`.

> 💡 Exécute les cellules **dans l'ordre**. Les sorties attendues de l'énoncé sont rappelées pour que tu puisses comparer.

### Installation (une seule fois)
On a besoin de `tiktoken`, le tokenizer officiel d'OpenAI.

In [ ]:
%pip install -q tiktoken

In [2]:
from importlib.metadata import version
import torch

print("torch    :", version("torch"))
print("tiktoken :", version("tiktoken"))

torch    : 2.14.1
tiktoken : 0.14.0


---
## Partie 1 — Comprendre le tokenizer

Un réseau de neurones ne manipule **que des nombres**. Il faut donc :
1. **découper** le texte en morceaux appelés **tokens** (mots, ponctuation...),
2. associer à chaque token un **numéro** (son *id*).

### Q1. Charger le texte
Si le fichier `the-verdict.txt` n'est pas dans le dossier, on le télécharge automatiquement.

In [3]:
import os
import urllib.request

file_path = "the-verdict.txt"
if not os.path.exists(file_path):
    url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/"
           "ch02/01_main-chapter-code/the-verdict.txt")
    urllib.request.urlretrieve(url, file_path)

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))   # attendu : 20479
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


### Q2. Tokenizer n°1 : couper sur les espaces

`re.split(motif, texte)` coupe le texte partout où le **motif** (une *expression régulière*) apparaît.
- `\s` = n'importe quel caractère d'espacement (espace, tabulation, retour à la ligne),
- les **parenthèses** `( )` autour du motif disent à Python de **garder** les séparateurs dans le résultat (d'où les `' '` dans la liste).

In [4]:
import re

result = re.split(r'(\s)', raw_text[:299])
print(result)

['I', ' ', 'HAD', ' ', 'always', ' ', 'thought', ' ', 'Jack', ' ', 'Gisburn', ' ', 'rather', ' ', 'a', ' ', 'cheap', ' ', 'genius--though', ' ', 'a', ' ', 'good', ' ', 'fellow', ' ', 'enough--so', ' ', 'it', ' ', 'was', ' ', 'no', ' ', 'great', ' ', 'surprise', ' ', 'to', ' ', 'me', ' ', 'to', ' ', 'hear', ' ', 'that,', ' ', 'in', ' ', 'the', ' ', 'height', ' ', 'of', ' ', 'his', ' ', 'glory,', ' ', 'he', ' ', 'had', ' ', 'dropped', ' ', 'his', ' ', 'painting,', ' ', 'married', ' ', 'a', ' ', 'rich', ' ', 'widow,', ' ', 'and', ' ', 'established', ' ', 'himself', ' ', 'in', ' ', 'a', ' ', 'villa', ' ', 'on', ' ', 'the', ' ', 'Riviera.', ' ', '(Though', ' ', 'I', ' ', 'rather', ' ', 'thought', ' ', 'it', ' ', 'would', ' ', 'h']


👀 Problème : la ponctuation reste collée aux mots (`'that,'`, `'Riviera.'`, `'genius--though'`). Donc `that` et `that,` seraient deux tokens différents ! On améliore ça.

### Q3. Tokenizer n°2 : couper aussi sur la ponctuation

Lecture du motif `([,.:;?_!"()\']|--|\s)` :
- `[,.:;?_!"()\']` → **un** de ces symboles de ponctuation,
- `|` signifie « **ou** »,
- `--` → la séquence « double tiret »,
- `\s` → un espace.

Ensuite on enlève les espaces autour (`item.strip()`) et on **jette les tokens vides** (`if item.strip()`).

In [5]:
def simple_split(text):
    morceaux = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    return [item.strip() for item in morceaux if item.strip()]

print(simple_split(raw_text[:299]))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'h']


### Q4. Nombre de tokens et nombre de tokens différents

`set(...)` retire les doublons → on obtient les tokens **uniques**.

In [6]:
preprocessed = simple_split(raw_text)

print("Nombre de tokens           :", len(preprocessed))        # 4690
print("Nombre de tokens différents :", len(set(preprocessed)))  # 1130

Nombre de tokens           : 4690
Nombre de tokens différents : 1130


### Q5. Créer le vocabulaire

Le **vocabulaire** est un dictionnaire `{token: numéro}`. On trie les tokens par ordre alphabétique (la ponctuation passe avant les lettres) puis on les numérote avec `enumerate`.

In [7]:
all_words = sorted(set(preprocessed))
vocab = {token: i for i, token in enumerate(all_words)}

print("Taille du vocabulaire :", len(vocab))
for item in list(vocab.items())[:16]:    # les 16 premières entrées
    print(item)

Taille du vocabulaire : 1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)


### Q6. La classe `SimpleTokenizerV1`

- `encode` : texte → liste d'ids (on découpe, puis on cherche chaque token dans `vocab`).
- `decode` : liste d'ids → texte. Il faut le **dictionnaire inverse** `{numéro: token}`.

Au décodage, on recolle les tokens avec des espaces, puis on supprime l'espace **avant** la ponctuation (`"you , know"` → `"you, know"`).

In [8]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab                                # token -> id
        self.int_to_str = {i: s for s, i in vocab.items()}     # id -> token

    def encode(self, text):
        tokens = simple_split(text)
        return [self.str_to_int[t] for t in tokens]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        # supprime l'espace devant la ponctuation : "you , know" -> "you, know"
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


tokenizer = SimpleTokenizerV1(vocab)

text = '"It\'s the last he painted, you know,"'
ids = tokenizer.encode(text)
print(ids)                       # attendu : [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1]
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1]
" It' s the last he painted, you know,"


> Remarque : l'énoncé affiche `[56, 2, 850, ...]` sans les guillemets `"` (id 1) au début et à la fin ; les ids du milieu sont bien identiques.
> Le décodage donne `It' s` au lieu de `It's` : notre tokenizer simple ne sait pas recoller l'apostrophe. Ce n'est pas grave, c'est attendu.

### Q7. Encoder une phrase avec un mot inconnu

In [9]:
try:
    tokenizer.encode("Hello, do you like tea. Is this-- a test?")
except KeyError as e:
    print("KeyError :", e)

KeyError : 'Hello'


👀 **Ce qui se passe** : le mot `Hello` n'apparaît jamais dans `the-verdict.txt`, il n'est donc **pas dans le vocabulaire** → `KeyError`. Un tokenizer doit savoir gérer les mots inconnus.

### Q8. `SimpleTokenizerV2` avec deux tokens spéciaux
- `<|endoftext|>` (id 1130) : sépare deux documents différents,
- `<|unk|>` (id 1131) : remplace **tout mot inconnu**.

Seule différence dans `encode` : si un token n'est pas dans le vocabulaire, on le remplace par `<|unk|>`.

In [10]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])     # ajoutés à la fin -> ids 1130 et 1131
vocab = {token: i for i, token in enumerate(all_tokens)}

print(list(vocab.items())[-2:])


class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        tokens = simple_split(text)
        # NOUVEAU : mot inconnu -> <|unk|>
        tokens = [t if t in self.str_to_int else "<|unk|>" for t in tokens]
        return [self.str_to_int[t] for t in tokens]

    def decode(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

[('<|endoftext|>', 1130), ('<|unk|>', 1131)]


⚠️ Petit piège : notre `simple_split` couperait `<|endoftext|>` en morceaux (`<`, `|`, ...) ? Non : `<` et `|` ne sont pas dans notre liste de ponctuation, donc `<|endoftext|>` reste **un seul token** car il est entouré d'espaces.

### Q9. Tester `SimpleTokenizerV2`

In [11]:
tokenizer = SimpleTokenizerV2(vocab)

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace."
ids = tokenizer.encode(text)
print(ids)                     # [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
print(tokenizer.decode(ids))   # '<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


### Q10. Le vrai tokenizer de GPT-2 : *Byte Pair Encoding* (BPE)

Le problème de `<|unk|>` : on **perd de l'information** (`Hello` et `palace` deviennent identiques).

Le **BPE** résout ça : il découpe les mots inconnus en **sous-mots** plus petits (voire en lettres). Ainsi, **aucun mot n'est jamais inconnu**. Par exemple `someunknownPlace` → `some` + `unknown` + `Place`.

In [12]:
import tiktoken

bpe = tiktoken.get_encoding("gpt2")
print("Taille du vocabulaire GPT-2 :", bpe.n_vocab)   # 50257

text = "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."
ids = bpe.encode(text, allowed_special={"<|endoftext|>"})
print(ids)
print(bpe.decode(ids))

Taille du vocabulaire GPT-2 : 50257
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [13]:
# Pour voir comment le BPE a découpé le texte : on décode chaque id séparément
print([bpe.decode([i]) for i in ids])

['Hello', ',', ' do', ' you', ' like', ' tea', '?', ' ', '<|endoftext|>', ' In', ' the', ' sun', 'lit', ' terr', 'aces', 'of', ' some', 'unknown', 'Place', '.']


---
## Partie 2 — Créer un Dataset et un DataLoader

### L'idée : apprendre à prédire le token suivant

GPT apprend une seule chose : **deviner le prochain token**. On prépare donc des paires :

```
entrée  : [ I    HAD   always  thought ]
cible   : [ HAD  always thought Jack   ]   ← la même fenêtre décalée de 1
```
Pour chaque position, la cible est **le token qui suit**.

On fait glisser une **fenêtre** de taille `max_length` sur le texte, en avançant de `stride` tokens à chaque fois.

### Q1 & Q2. La classe `GPTDatasetV1`

In [14]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 1. Encoder tout le texte
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 2. Fenêtre glissante
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk  = token_ids[i     : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]   # décalé de 1
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

### Q3. La fonction `create_dataloader_v1`
Elle regroupe les 3 étapes : tokenizer → dataset → dataloader.

In [15]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")                        # 1. tokenizer BPE
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)       # 2. dataset
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,  # 3. dataloader
                      drop_last=drop_last, num_workers=num_workers)

### Q4. Tester avec `max_length = stride = 4`

Quand `stride == max_length`, les fenêtres **ne se chevauchent pas** : chaque token n'apparaît qu'une seule fois en entrée, ce qui limite le sur-apprentissage.

In [16]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)          # un itérateur sur les lots
inputs, targets = next(data_iter)     # le premier lot

print("Inputs:\n", inputs)
print("\nTargets:\n", targets)
print("\nShape :", inputs.shape)      # (8, 4) = 8 séquences de 4 tokens

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])

Shape : torch.Size([8, 4])


👀 Vérifie : chaque ligne de `Targets` est la ligne de `Inputs` décalée d'un cran vers la gauche (avec un nouveau token à la fin).

---
## Partie 3 — Les embeddings de mots

Un id comme `2885` n'a pas de « sens » numérique (le token 2885 n'est pas « plus grand » que le token 40). On associe donc à chaque token un **vecteur** de taille `m` que le modèle **apprendra** : c'est l'**embedding**.

Concrètement c'est une **matrice** `E` de taille `n × m` (n = taille du vocabulaire) : **la ligne `i` est le vecteur du token `i`**.

> Mathématiquement : multiplier le vecteur *one-hot* `[0,…,1,…,0]` par `E` revient exactement à **lire la ligne `i`** de `E`. `nn.Embedding` fait directement cette lecture, c'est plus rapide.

### Q1. Une petite couche d'embedding (n = 6, m = 3)

In [17]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(num_embeddings=6, embedding_dim=3)
print(embedding_layer.weight)    # 6 lignes (une par token) de 3 valeurs

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


### Q2. Embedding du token d'id 0
On obtient bien la **ligne 0** de la matrice.

In [18]:
print(embedding_layer(torch.tensor([0])))

tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)


### Q3. Plusieurs tokens d'un coup
Pour `[2, 3, 5, 1]`, on récupère les lignes 2, 3, 5 et 1.

In [19]:
print(embedding_layer(torch.tensor([2, 3, 5, 1])))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


### Q4. La couche d'embedding de GPT-2 (n = 50257, m = 256)

On envoie le premier lot (8 séquences de 4 tokens). Chaque token devient un vecteur de 256 nombres :
**(8, 4) → (8, 4, 256)**.

In [20]:
vocab_size = 50257
output_dim = 256

torch.manual_seed(123)
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

token_embeddings = token_embedding_layer(inputs)
print(token_embeddings)
print("Shape :", token_embeddings.shape)   # torch.Size([8, 4, 256])

tensor([[[-6.3964e-02,  3.3174e-01,  1.0698e-01,  ...,  5.3491e-01,
          -8.0244e-01, -2.3238e+00],
         [-3.5248e-01,  3.5087e-01,  9.8728e-01,  ..., -1.8466e+00,
          -1.7034e+00,  3.2226e-01],
         [ 1.0017e+00,  9.2986e-01, -1.2633e+00,  ..., -1.2256e+00,
           1.1179e+00,  1.3427e-01],
         [ 7.9961e-01,  2.2837e+00, -6.5249e-01,  ..., -1.1217e+00,
           4.7057e-01,  1.5314e-01]],

        [[-1.0823e-01, -1.2723e+00, -1.2217e+00,  ..., -9.1995e-01,
           2.0073e+00, -1.4138e+00],
         [-2.4273e-01,  9.1447e-01,  1.0885e+00,  ..., -8.6509e-01,
           3.5269e+00,  7.2247e-01],
         [-5.4342e-01,  1.6203e+00,  1.2222e+00,  ...,  6.8139e-01,
          -1.4032e+00,  1.4922e-01],
         [-3.5035e-01, -9.3247e-01, -1.2900e+00,  ..., -1.4980e+00,
           1.3996e-01,  3.7301e-01]],

        [[ 6.7677e-02, -1.0757e+00,  1.3706e-01,  ..., -3.9116e-01,
          -1.0974e+00,  7.5906e-01],
         [-1.2561e+00,  3.1799e-01,  2.0510e-02,  .

---
## Partie 4 — Les embeddings de position

Problème : dans `token_embeddings`, le mot « chat » a **le même vecteur** qu'il soit en 1ère ou en 4ème position. Or l'ordre des mots compte !

Solution de GPT-2 (*position absolue*) : une **deuxième** table d'embedding, cette fois **une ligne par position** (0, 1, 2, ..., l-1). On **additionne** ensuite le vecteur du mot et le vecteur de sa position.

### Q1. Couche d'embedding de position (l = 4, m = 256)

In [21]:
context_length = 4     # le modèle accepte au maximum 4 tokens

pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
print(pos_embedding_layer.weight.shape)   # (4, 256)
print(pos_embedding_layer.weight)

torch.Size([4, 256])
Parameter containing:
tensor([[ 0.5423, -0.1224, -1.4150,  ...,  0.2515, -2.3067,  0.8155],
        [-0.3973, -1.2575, -1.9800,  ..., -0.1207,  0.3075, -0.6422],
        [ 0.1840,  1.1128,  1.0052,  ...,  0.2081,  0.5531, -1.1619],
        [ 1.4155,  0.6599,  0.3760,  ...,  0.7034, -0.6108,  0.1080]],
       requires_grad=True)


### Q2. Additionner token + position

- `torch.arange(4)` = `[0, 1, 2, 3]` → les 4 positions → shape `(4, 256)`.
- On additionne un tenseur `(8, 4, 256)` et un tenseur `(4, 256)` : PyTorch **répète automatiquement** le second pour les 8 séquences (c'est le *broadcasting*).

La shape **ne change pas** : `(8, 4, 256)`. Ce sont ces vecteurs qui entreront dans l'attention.

In [22]:
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("pos_embeddings   :", pos_embeddings.shape)     # (4, 256)

input_embeddings = token_embeddings + pos_embeddings
print("input_embeddings :", input_embeddings.shape)   # (8, 4, 256)

pos_embeddings   : torch.Size([4, 256])
input_embeddings : torch.Size([8, 4, 256])


---
## Partie 5 — Comprendre le mécanisme d'attention (à la main)

### L'intuition
Dans « *Your journey starts with one step* », le sens de **journey** dépend des autres mots. L'attention calcule pour **journey** un nouveau vecteur qui est un **mélange pondéré** des vecteurs de tous les mots, en donnant plus de poids aux mots **les plus pertinents**.

Pour ça, chaque mot produit 3 vecteurs (avec 3 matrices apprises) :

| | Question qu'il pose | Analogie (moteur de recherche) |
|---|---|---|
| **Query** (requête) | « qu'est-ce que je cherche ? » | ce que tu tapes dans la barre |
| **Key** (clé) | « qu'est-ce que je contiens ? » | le titre de chaque page |
| **Value** (valeur) | « quelle info je transmets ? » | le contenu de la page |

Recette : **score** = query · key → **poids** = softmax(score / √d) → **résultat** = Σ poids × value.

In [23]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],   # Your     (x1)
   [0.55, 0.87, 0.66],   # journey  (x2)
   [0.57, 0.85, 0.64],   # starts   (x3)
   [0.22, 0.58, 0.33],   # with     (x4)
   [0.77, 0.25, 0.10],   # one      (x5)
   [0.05, 0.80, 0.55]]   # step     (x6)
)
print(inputs.shape)   # 6 mots, chacun représenté par 3 nombres

torch.Size([6, 3])


### Q1. Les trois matrices `W_query`, `W_key`, `W_value` (taille 3 × d_out)
`requires_grad=False` ici car on ne fait que de la démonstration (pas d'entraînement).

In [24]:
d_in = 3      # taille des vecteurs d'entrée
d_out = 2     # taille voulue en sortie

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

print("W_query", W_query)
print("W_key", W_key)
print("W_value", W_value)

W_query Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
W_key Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
W_value Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


### Q2. La query de « journey »

In [25]:
x_2 = inputs[1]              # "journey" (indice 1 car on compte à partir de 0)
query_2 = x_2 @ W_query
print(query_2)               # tensor([0.4306, 1.4551])

tensor([0.4306, 1.4551])


### Q3. Les keys et values de **tous** les mots
`(6 × 3) @ (3 × 2) = (6 × 2)` : une key et une value par mot.

In [26]:
keys   = inputs @ W_key
values = inputs @ W_value
print("keys.shape   :", keys.shape)     # (6, 2)
print("values.shape :", values.shape)   # (6, 2)

keys.shape   : torch.Size([6, 2])
values.shape : torch.Size([6, 2])


### Q4. Les scores d'attention de « journey »
Produit scalaire de la query de « journey » avec la key de chaque mot : **plus le score est grand, plus le mot est pertinent pour journey**.

In [27]:
attn_scores_2 = query_2 @ keys.T     # (2) @ (2 x 6) -> (6)
print(attn_scores_2)                 # tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


### Q5. Normaliser : diviser par √d_out puis softmax
- Le **softmax** transforme les scores en poids positifs qui **somment à 1** (des pourcentages).
- La division par **√d_out** évite que les scores soient trop grands : sinon le softmax deviendrait « tout ou rien » (un poids à 1, les autres à 0) et les gradients seraient minuscules.

In [28]:
d_k = keys.shape[-1]    # = d_out = 2
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)                   # tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
print("Somme :", attn_weights_2.sum())  # 1

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
Somme : tensor(1.)


### Q6. Le nouveau vecteur de « journey » (le *context vector*)
C'est la **moyenne pondérée** des values.

In [29]:
context_vec_2 = attn_weights_2 @ values   # (6) @ (6 x 2) -> (2)
print(context_vec_2)                       # tensor([0.3061, 0.8210])

tensor([0.3061, 0.8210])


---
## Partie 6 — La classe `SelfAttention`

On fait **exactement le même calcul**, mais pour **tous les mots en même temps** grâce aux matrices :
- `queries = inputs @ W_query` → une query par mot,
- `attn_scores = queries @ keys.T` → une matrice 6 × 6 (le score de chaque mot avec chaque mot),
- softmax ligne par ligne, puis `@ values`.

On remplace les matrices par des `nn.Linear(..., bias=False)` : c'est la même chose (une multiplication par une matrice de poids) mais avec une meilleure initialisation.

### Q1 & Q2.

In [30]:
import torch.nn as nn


class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        queries = self.W_query(x)          # (6, d_out)
        keys    = self.W_key(x)            # (6, d_out)
        values  = self.W_value(x)          # (6, d_out)

        attn_scores  = queries @ keys.T    # (6, 6)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec  = attn_weights @ values   # (6, d_out)
        return context_vec


torch.manual_seed(789)
sa = SelfAttention(d_in=3, d_out=2)
print(sa(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


### Q3. Le masque causal : interdire de regarder le futur

GPT génère le texte **de gauche à droite**. Quand il traite « journey », il ne doit **pas voir** « starts », « with »... (sinon il tricherait : il verrait le mot qu'il doit prédire !).

On met donc la partie **au-dessus de la diagonale** de la matrice des scores à **−∞** avant le softmax. Comme $e^{-\infty} = 0$, ces mots auront un poids de **0**.

```
           Your journey starts ...
Your     [  ok   -inf   -inf  ...]
journey  [  ok    ok    -inf  ...]
starts   [  ok    ok     ok   ...]
```
- `torch.triu(..., diagonal=1)` : met des 1 au-dessus de la diagonale (le « futur »),
- `masked_fill_(mask, -inf)` : remplace les cases où `mask` est vrai par −∞ (le `_` final = modification **sur place**).

In [31]:
# On recalcule les scores avec la couche sa (pour l'exemple)
queries = sa.W_query(inputs)
keys    = sa.W_key(inputs)
attn_scores = queries @ keys.T

n = inputs.shape[0]   # 6 mots
mask = torch.triu(torch.ones(n, n), diagonal=1).bool()
print("Masque (True = futur à cacher) :\n", mask)

attn_scores.masked_fill_(mask, -torch.inf)
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print("\nPoids d'attention masqués :\n", attn_weights)

Masque (True = futur à cacher) :
 tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

Poids d'attention masqués :
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


👀 Chaque ligne somme toujours à 1, et tout ce qui est au-dessus de la diagonale vaut 0. Le 1er mot ne peut regarder que lui-même (poids = 1).

### Q4. La classe `CausalAttention` (masque + dropout + batch)

Nouveautés par rapport à `SelfAttention` :
1. **Batch** : l'entrée a maintenant la forme `(b, n, d_in)` (b séquences). On utilise `keys.transpose(1, 2)` au lieu de `.T` pour ne transposer que les 2 dernières dimensions.
2. **Masque causal** : créé une seule fois dans `__init__` avec `register_buffer` (le masque fait partie du modèle et suit le `.to(device)`, mais ce n'est **pas** un paramètre à apprendre). Dans `forward`, on ne prend que la partie utile `[:n, :n]`.
3. **Dropout** : pendant l'entraînement, met au hasard certains poids d'attention à 0, pour éviter que le modèle ne dépende trop de quelques mots (lutte contre le sur-apprentissage).

In [32]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, n, d_in = x.shape                         # batch, nb de tokens, taille d'entrée

        queries = self.W_query(x)                    # (b, n, d_out)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)               # (b, n, n)
        attn_scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values                        # (b, n, d_out)
        return context_vec

### Q5. Tester avec un batch de 2 séquences
`torch.stack` empile deux fois `inputs` → shape `(2, 6, 3)`.

In [33]:
batch = torch.stack((inputs, inputs), dim=0)
print("batch.shape :", batch.shape)   # (2, 6, 3)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0)

context_vecs = ca(batch)
print(context_vecs)
print("Shape :", context_vecs.shape)  # (2, 6, 2)

batch.shape : torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
Shape : torch.Size([2, 6, 2])


---
## Partie 7 — L'attention multi-tête (`MultiHeadAttention`)

### L'intuition
Une seule « tête » d'attention ne regarde les relations entre mots que **d'une seule façon**. Avec **plusieurs têtes**, chacune peut se spécialiser (une sur la grammaire, une sur le sens, une sur les mots proches...). On **concatène** ensuite leurs résultats.

### L'astuce d'implémentation
Plutôt que de créer `num_heads` couches `CausalAttention` séparées, on calcule **une seule grosse** projection de taille `d_out`, puis on la **découpe** en `num_heads` morceaux de taille `head_dim = d_out / num_heads`.

Suivi des formes (b = batch, n = nb de tokens, h = nb de têtes, hd = head_dim) :

| Étape | Forme |
|---|---|
| `queries = W_query(x)` | `(b, n, d_out)` |
| `.view(b, n, h, hd)` — découpe en têtes | `(b, n, h, hd)` |
| `.transpose(1, 2)` — les têtes passent devant | `(b, h, n, hd)` |
| `queries @ keys.transpose(2, 3)` — scores | `(b, h, n, n)` |
| `attn_weights @ values` | `(b, h, n, hd)` |
| `.transpose(1, 2)` | `(b, n, h, hd)` |
| `.contiguous().view(b, n, d_out)` — on recolle les têtes | `(b, n, d_out)` |
| `out_proj` — couche linéaire qui mélange les têtes | `(b, n, d_out)` |

> `contiguous()` : après un `transpose`, les nombres ne sont plus rangés dans l'ordre en mémoire, et `view` refuse de fonctionner (cf. Practical 0, Q4). `contiguous()` les réordonne.

In [34]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out doit être divisible par num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads          # taille de chaque tête

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)     # mélange les sorties des têtes
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, n, d_in = x.shape

        # 1. Projections : (b, n, d_out)
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        # 2. Découpage en têtes : (b, n, d_out) -> (b, n, h, hd) -> (b, h, n, hd)
        queries = queries.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        keys    = keys.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        values  = values.view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Attention causale dans chaque tête : (b, h, n, n)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 4. Combinaison avec les values : (b, h, n, hd) -> (b, n, h, hd)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 5. On recolle les têtes : (b, n, d_out)
        context_vec = context_vec.contiguous().view(b, n, self.d_out)

        # 6. Projection finale
        return self.out_proj(context_vec)

### Q2. Tester avec `d_out = 2`, `num_heads = 2`, `dropout = 0.0`

In [35]:
torch.manual_seed(123)
b, context_length, d_in = batch.shape
mha = MultiHeadAttention(d_in=d_in, d_out=2, context_length=context_length,
                         dropout=0.0, num_heads=2)

context_vecs = mha(batch)
print(context_vecs)
print("Shape :", context_vecs.shape)   # (2, 6, 2)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
Shape : torch.Size([2, 6, 2])


---
## 🎯 Résumé de l'essentiel

| Étape | Ce qu'elle fait | Forme |
|---|---|---|
| **Tokenizer (BPE)** | texte → ids ; découpe les mots inconnus en sous-mots | `(b, n)` |
| **Dataset** | fenêtre glissante ; cible = entrée décalée de 1 | `(b, n)` et `(b, n)` |
| **Token embedding** | id → vecteur appris (lecture d'une ligne de matrice) | `(b, n, m)` |
| **Position embedding** | position → vecteur appris, **additionné** au précédent | `(b, n, m)` |
| **Self-attention** | chaque mot = moyenne pondérée des values ; poids = softmax(Q·Kᵀ/√d) | `(b, n, d_out)` |
| **Masque causal** | −∞ au-dessus de la diagonale : pas d'accès au futur | |
| **Multi-head** | plusieurs têtes en parallèle (view + transpose), puis `out_proj` | `(b, n, d_out)` |

La formule à retenir :

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + \text{masque}\right) V$$